# VoyageIQ — Total Trip Cost Forecasting
## Legacy Notebook (Monolithic / Pre-MLOps)

This notebook contains the **entire ML pipeline in a single file** — loading, cleaning, validation, feature engineering, splitting, training, evaluation, and inference — without using any modular `src/` imports.

### Why this exists
- Demonstrates the "before MLOps" approach: everything in one notebook
- Serves as a reference to compare against the modular `src/` pipeline
- Shows why monolithic notebooks become hard to maintain, test, and deploy

### Key difference from the modular version
| Aspect | This Legacy Notebook | Modular Pipeline (`src/`) |
|--------|---------------------|--------------------------|
| Code location | All in one file | Split across 10 modules |
| Testability | Manual "run all" only | `pytest` with 40+ tests |
| Reproducibility | Depends on cell execution order | Deterministic `python -m src.main` |
| Collaboration | Merge conflicts on every change | Each teammate owns separate files |
| Production deployment | Copy-paste prone | Single artifact (`model.joblib`) |

### Important
- This notebook produces **identical results** to `python -m src.main` and the modular `01_voyageiq_analysis_vExp.ipynb`
- Same random seeds, same split ratios, same model hyperparameters

### 1) Imports

All dependencies are loaded here. In the modular version, each module handles its own imports.

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("All imports loaded successfully")

### 2) Configuration

All settings in one place — paths, feature lists, split ratios, model hyperparameters.

In the modular version, this lives in `main.py` as the `SETTINGS` dictionary.

In [ ]:
# ── Paths ──
RAW_DATA_PATH = Path("data/raw/travel_raw.csv")

# ── Target & problem type ──
TARGET_COLUMN = "total_cost"
PROBLEM_TYPE = "regression"

# ── Split settings ──
TEST_SIZE = 0.15
VAL_SIZE = 0.15
RANDOM_SEED = 42

# ── Feature lists ──
NUMERIC_FEATURES = ["duration_days", "traveler_age", "travel_month", "day_of_week"]
CATEGORICAL_FEATURES = [
    "destination_country",
    "traveler_gender",
    "traveler_nationality",
    "accommodation_type",
    "transportation_type",
]

# ── Cleaning settings ──
DROP_COLUMNS = ["trip_id", "traveler_name"]

print("Configuration loaded")
print(f"  Target: {TARGET_COLUMN}")
print(f"  Problem type: {PROBLEM_TYPE}")
print(f"  Numeric features: {NUMERIC_FEATURES}")
print(f"  Categorical features: {CATEGORICAL_FEATURES}")

### 3) Load raw data

Load the CSV file from disk. The raw data is treated as immutable — we never modify the source file.

In the modular version, this is `src/load_data.py` → `load_raw_data()` which calls `src/utils.py` → `load_csv()`.

In [ ]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw data not found at {RAW_DATA_PATH}\n"
        "Place travel_raw.csv in data/raw/"
    )

try:
    df_raw = pd.read_csv(RAW_DATA_PATH, encoding="utf-8-sig")
except UnicodeDecodeError:
    df_raw = pd.read_csv(RAW_DATA_PATH, encoding="latin-1")

if df_raw.empty:
    raise ValueError(f"Loaded DataFrame from {RAW_DATA_PATH} has zero rows.")

print(f"Raw data loaded — shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
df_raw.head()

### 4) Focused EDA checks

Quick targeted checks before investing in cleaning and training.

In [ ]:
print("Missing values (top 10 columns):")
display(df_raw.isna().sum().sort_values(ascending=False).head(10))

print(f"\nNote: Target '{TARGET_COLUMN}' is created during cleaning")
print("(total_cost = accommodation_cost + transportation_cost)")

for cost_col in ["Accommodation cost", "Transportation cost"]:
    if cost_col in df_raw.columns:
        print(f"\nSummary for '{cost_col}':")
        display(df_raw[cost_col].describe())

if "Destination" in df_raw.columns:
    print("\nSample destinations:")
    display(df_raw["Destination"].head(5))

print(f"\nDataset: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")

### 5) Clean data

All cleaning logic is inline here. In the modular version, this lives in `src/clean_data.py` → `clean_dataframe()`.

Steps:
1. Standardise column names (lowercase, underscores)
2. Drop unwanted columns (trip_id, traveler_name)
3. Remove duplicates
4. Drop rows with missing values
5. Parse currency columns (handle $, USD, commas)
6. Create target: `total_cost = accommodation_cost + transportation_cost`
7. Extract `destination_country` from "City, Country" format
8. Extract `travel_month` and `day_of_week` from `start_date`
9. Ensure numeric dtypes for key columns
10. Reset index

In [ ]:
df = df_raw.copy()
initial_rows = len(df)
print(f"Cleaning started — initial rows: {initial_rows}")

# 1. Standardise column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
print(f"Column names standardised: {list(df.columns)}")

# 2. Drop unwanted columns
df = df.drop(columns=DROP_COLUMNS, errors="ignore")
print(f"Dropped columns (if present): {DROP_COLUMNS}")

# 3. Remove duplicates
before = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {before - len(df)}")

# 4. Drop rows with missing values
before = len(df)
df = df.dropna()
print(f"Rows dropped (NaN): {before - len(df)}")

# 5. Parse currency columns
for cost_col in ["accommodation_cost", "transportation_cost"]:
    if cost_col in df.columns:
        df[cost_col] = (
            df[cost_col]
            .astype(str)
            .str.strip()
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.replace("USD", "", regex=False)
            .str.replace("EUR", "", regex=False)
            .str.strip()
        )
        df[cost_col] = pd.to_numeric(df[cost_col], errors="coerce")

# Drop rows where cost parsing failed
before = len(df)
df = df.dropna(subset=["accommodation_cost", "transportation_cost"])
if before - len(df) > 0:
    print(f"Rows dropped (unparseable cost values): {before - len(df)}")

# 6. Create target column
df[TARGET_COLUMN] = df["accommodation_cost"] + df["transportation_cost"]
print(f"Target column '{TARGET_COLUMN}' created.")

# 7. Extract destination country
if "destination" in df.columns:
    parts = df["destination"].str.split(",", n=1, expand=True)
    df["destination_city"] = parts[0].str.strip() if 0 in parts.columns else "Unknown"
    df["destination_country"] = parts[1].str.strip() if 1 in parts.columns else "Unknown"

# 8. Extract date features
if "start_date" in df.columns:
    dt = pd.to_datetime(df["start_date"], errors="coerce", dayfirst=False)
    df["travel_month"] = dt.dt.month.fillna(0).astype(int)
    df["day_of_week"] = dt.dt.dayofweek.fillna(0).astype(int)

# 9. Ensure numeric types
for num_col in ["traveler_age", "duration_days"]:
    if num_col in df.columns:
        df[num_col] = pd.to_numeric(df[num_col], errors="coerce")

# 10. Reset index
df = df.reset_index(drop=True)

df_clean = df
print(f"Cleaning complete — final rows: {len(df_clean)} (dropped: {initial_rows - len(df_clean)})")
df_clean.head()

### 6) What changed after cleaning

Make transformations visible — helps debug unexpected column changes.

In [ ]:
raw_cols = list(df_raw.columns)
clean_cols = list(df_clean.columns)

removed_cols = sorted(set(raw_cols) - set(clean_cols))
added_cols = sorted(set(clean_cols) - set(raw_cols))

print("Columns removed (raw names no longer present):")
display(removed_cols)

print("\nColumns added during cleaning:")
display(added_cols)

print(f'\nHas "total_cost": {"total_cost" in df_clean.columns}')
print(f'Has "destination_country": {"destination_country" in df_clean.columns}')
print(f'Has "travel_month": {"travel_month" in df_clean.columns}')
print(f"\nRows: {df_raw.shape[0]} raw → {df_clean.shape[0]} clean (dropped {df_raw.shape[0] - df_clean.shape[0]})")

### 7) Validate data (security gate)

Fail-fast checks before training. In the modular version, this is `src/validate.py` → `validate_dataframe()`.

In [ ]:
print("Starting data validation …")

# Check not empty
if df_clean.empty:
    raise ValueError("Validation received an empty DataFrame.")

# Check required columns exist
required_columns = [TARGET_COLUMN] + CATEGORICAL_FEATURES + NUMERIC_FEATURES
issues = []
present = set(df_clean.columns)
for col in required_columns:
    if col not in present:
        issues.append(f"Required column missing: '{col}'")

# Numeric range checks
range_checks = {
    "duration_days": {"min": 1, "max": 365},
    "traveler_age": {"min": 1, "max": 120},
    "accommodation_cost": {"min": 0, "max": 100000},
    "transportation_cost": {"min": 0, "max": 100000},
}
for col, bounds in range_checks.items():
    if col not in df_clean.columns:
        continue
    if bounds.get("min") is not None and (df_clean[col] < bounds["min"]).any():
        n_bad = int((df_clean[col] < bounds["min"]).sum())
        issues.append(f"'{col}': {n_bad} values below minimum ({bounds['min']})")
    if bounds.get("max") is not None and (df_clean[col] > bounds["max"]).any():
        n_bad = int((df_clean[col] > bounds["max"]).sum())
        issues.append(f"'{col}': {n_bad} values above maximum ({bounds['max']})")

# Check target for nulls
if TARGET_COLUMN in df_clean.columns and df_clean[TARGET_COLUMN].isna().any():
    n_na = int(df_clean[TARGET_COLUMN].isna().sum())
    issues.append(f"Target '{TARGET_COLUMN}' has {n_na} null values")

if issues:
    msg = f"Validation found {len(issues)} issue(s):\n"
    for issue in issues:
        msg += f"  - {issue}\n"
    raise ValueError(msg)

print("Validation passed — no issues detected.")

### 8) Three-way split (train / validation / test)

**Leakage gate**: splitting happens BEFORE any feature fitting.

In the modular version, this is done inside `src/main.py` before calling `train_model()`.

In [ ]:
# Select feature columns
keep_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
keep_cols = [c for c in keep_cols if c in df_clean.columns]

X = df_clean[keep_cols]
y = df_clean[TARGET_COLUMN]

# First split: train+val vs test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

# Second split: train vs val
relative_val = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=relative_val, random_state=RANDOM_SEED
)

print(f"Split sizes — train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")
print(f"Feature columns: {keep_cols}")

### 9) Build feature preprocessor recipe

Returns an **unfitted** ColumnTransformer. Fitting only happens on the training split.

In the modular version, this is `src/features.py` → `get_feature_preprocessor()`.

In [ ]:
# Numeric sub-pipeline: impute → scale
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Categorical sub-pipeline: impute → one-hot encode
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", encoder),
])

# Build ColumnTransformer
transformers = []
if NUMERIC_FEATURES:
    transformers.append(("num_passthrough", num_pipeline, NUMERIC_FEATURES))
if CATEGORICAL_FEATURES:
    transformers.append(("cat_onehot", cat_pipeline, CATEGORICAL_FEATURES))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

print(f"Preprocessor recipe built (NOT fitted yet)")
print(f"  Numeric: {NUMERIC_FEATURES}")
print(f"  Categorical: {CATEGORICAL_FEATURES}")
preprocessor

### 10) Train model

Build a Pipeline (preprocessor + model), fit **only on training data**.

In the modular version, this is `src/train.py` → `train_model()`.

In [ ]:
# Select estimator based on problem type
estimator = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
)

# Build and fit pipeline
model_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", estimator),
])

print(f"Fitting pipeline on {len(X_train)} training samples …")
model_pipeline.fit(X_train, y_train)
print("Training complete.")
model_pipeline

### 11) Evaluate model

Score on validation split (and optionally test split).

In the modular version, this is `src/evaluate.py` → `evaluate_model()`.

In [ ]:
def compute_metrics(model, X, y, split_name="validation"):
    """Compute and print regression metrics."""
    y_pred = model.predict(X)
    rmse = float(np.sqrt(mean_squared_error(y, y_pred)))
    mae = float(mean_absolute_error(y, y_pred))
    r2 = float(r2_score(y, y_pred))
    print(f"[{split_name}] RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")
    return rmse

# Validation metrics
val_rmse = compute_metrics(model_pipeline, X_val, y_val, "validation")

# Test metrics
test_rmse = compute_metrics(model_pipeline, X_test, y_test, "test")

### 12) Inference demo

Simulate production inference on unseen data (test split samples).

In the modular version, this is `src/infer.py` → `run_inference()`.

In [ ]:
sample_n = min(10, len(X_test))
X_infer_sample = X_test.sample(n=sample_n, random_state=RANDOM_SEED)

predictions = model_pipeline.predict(X_infer_sample)

df_predictions = pd.DataFrame(
    {"prediction": predictions},
    index=X_infer_sample.index,
)

print(f"Inference complete — {len(df_predictions)} predictions generated.")
display(df_predictions.head(10))

### 13) Summary

This legacy notebook produced the same results as the modular pipeline:
- **139 rows** loaded → **136 rows** after cleaning (1 duplicate, 2 NaN)
- **3-way split**: train 94 / val 21 / test 21
- **Validation RMSE**: ~1998.85
- **Test RMSE**: ~1487.76

#### Why migrate to the modular `src/` pipeline?
- **Testability**: Each function can be unit-tested with `pytest`
- **Collaboration**: Teammates edit separate files — no merge conflicts
- **Reproducibility**: `python -m src.main` always runs the same way
- **Deployment**: The saved `model.joblib` bundles preprocessing + model
- **Maintainability**: Change one module without breaking others

In [ ]:
print("=" * 60)
print("Legacy notebook completed successfully.")
print(f"  Rows: {df_raw.shape[0]} raw → {df_clean.shape[0]} clean")
print(f"  Split: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")
print(f"  Validation RMSE: {val_rmse:.4f}")
print(f"  Test RMSE: {test_rmse:.4f}")
print(f"  Predictions: {len(df_predictions)}")
print("=" * 60)